# 🏛️ Notebook 09: Modern Architectures

LLaMA (RMSNorm, SwiGLU, RoPE, GQA), Mistral (sliding window attention), and Gemma architecture differences.

**Prerequisites:** Notebooks 01-08

In [ ]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

## Architecture Evolution Timeline

| Year | Model | Key Innovation |
|------|-------|---------------|
| 2017 | Transformer | Self-attention, sinusoidal PE |
| 2018 | GPT-1 | Decoder-only, pre-training |
| 2019 | GPT-2 | Larger scale, learned PE |
| 2020 | GPT-3 | 175B params, few-shot learning |
| 2023 | LLaMA | RMSNorm, SwiGLU, RoPE, GQA |
| 2023 | Mistral | Sliding window attention |
| 2024 | Gemma 2 | GQA, logit soft-capping |
| 2025 | Gemma 4 | Multimodal, improved efficiency |

## LLaMA Architecture

Key differences from vanilla transformer:
1. **RMSNorm** instead of LayerNorm (pre-norm placement)
2. **SwiGLU FFN** instead of standard FFN with ReLU/GELU
3. **RoPE** instead of learned/sinusoidal positional embeddings
4. **GQA** instead of standard multi-head attention

### Why Each Component Matters (Plain English)

You've built all of these components individually in earlier notebooks. Here's *why* LLaMA chose each one — explained without any math.

💡 **RMSNorm** — Think of normalization like adjusting the volume on a speaker. LayerNorm computes both the mean *and* the spread of activations, then corrects both. RMSNorm skips the mean calculation and only adjusts the spread (the "root mean square"). This is simpler, faster, and in practice works just as well. Fewer operations per layer means faster training across billions of tokens.

💡 **SwiGLU** — A standard activation like ReLU is a simple on/off switch: positive values pass through, negatives get zeroed out. SwiGLU is more like a smart filter that decides *how much* of each piece of information to pass through. It multiplies two parallel paths — one that transforms the data and one that *gates* it (controls the flow). This gating lets the network learn richer representations than a plain switch ever could.

💡 **RoPE (Rotary Position Embeddings)** — Older models added a position signal to each token's embedding, like stamping a number on it. RoPE instead *rotates* the embedding vector by an angle that depends on position — picture a clock hand that tells you what time (position) it is. The clever part: when two tokens compute attention, their relative rotation naturally encodes how far apart they are. This makes RoPE generalize better to sequence lengths the model never saw during training.

💡 **GQA (Grouped Query Attention)** — In standard multi-head attention, every query head has its own dedicated key and value head. That's expensive in memory, especially during inference when you cache all those K/V tensors. GQA is like a classroom where students (query heads) share textbooks (K/V heads) to save money (memory). For example, LLaMA 3 70B uses 64 query heads but only 8 K/V heads — an 8× reduction in KV-cache size with minimal quality loss.

💡 **No Bias Terms** — Classic neural network layers include a bias vector (an additive offset). Modern LLMs like LLaMA remove it entirely: `bias=False` on every linear layer. Fewer parameters means faster training and less memory, and empirically the model learns just fine without them. When you have billions of parameters, those small bias vectors add up — and they simply aren't needed.

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import math

# LLaMA-style block (already built in previous notebooks)
print('LLaMA Architecture Components:')
print('  ✅ RMSNorm (Notebook 06)')
print('  ✅ SwiGLU FFN (Notebook 06)')
print('  ✅ RoPE (Notebook 04)')
print('  ✅ GQA (Notebook 05)')
print()
print('LLaMA vs Vanilla Transformer:')
print(f"  {'Component':<20} {'Vanilla':<20} {'LLaMA'}")
print(f"  {'Normalization':<20} {'LayerNorm (post)':<20} {'RMSNorm (pre)'}")
print(f"  {'FFN activation':<20} {'ReLU/GELU':<20} {'SwiGLU'}")
print(f"  {'Position encoding':<20} {'Sinusoidal/Learned':<20} {'RoPE'}")
print(f"  {'Attention':<20} {'MHA':<20} {'GQA'}")
print(f"  {'Bias terms':<20} {'Yes':<20} {'No (bias=False)'}")
print(f"  {'Vocab embedding':<20} {'Separate':<20} {'Tied (input=output)'}")


## Mistral: Sliding Window Attention

Mistral uses **sliding window attention** where each token only attends to the W nearest tokens instead of all previous tokens. This reduces attention complexity from O(n²) to O(n×W).

💡 With W=4096, a 32K context model uses 8× less attention memory than full attention.

In [ ]:
def sliding_window_mask(seq_len: int, window_size: int) -> mx.array:
    """Create a sliding window causal mask.
    
    Each token attends only to the W nearest preceding tokens (including itself).
    Uses vectorized ops — no Python loops over positions.
    """
    rows = mx.arange(seq_len).reshape(-1, 1)  # query positions
    cols = mx.arange(seq_len).reshape(1, -1)  # key positions
    causal = rows >= cols                       # lower triangular (causal)
    in_window = (rows - cols) < window_size     # within sliding window
    mask = (causal & in_window).astype(mx.float32)
    mx.eval(mask)
    return mask

import numpy as np
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
seq = 16
full_mask = np.array(mx.tril(mx.ones((seq, seq))))
sw_mask = np.array(sliding_window_mask(seq, 4))

ax1.imshow(full_mask, cmap='Blues')
ax1.set_title('Full Causal Mask')
ax2.imshow(sw_mask, cmap='Blues')
ax2.set_title('Sliding Window (W=4)')
for ax in [ax1, ax2]:
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
plt.tight_layout()
plt.show()
print('⚡ Sliding window reduces memory from O(n²) to O(n×W)')

## Gemma Architecture

Gemma (Google) evolution:
- **Gemma 1**: Standard transformer with multi-query attention
- **Gemma 2**: GQA, logit soft-capping, interleaved local/global attention
- **Gemma 4**: Multimodal, improved efficiency

Key innovation: **logit soft-capping** — prevents attention logits from growing too large by applying `tanh(logits/cap) * cap`.

In [ ]:
def soft_cap(logits, cap=50.0):
    """Gemma 2 logit soft-capping."""
    return mx.tanh(logits / cap) * cap

# Demo
logits = mx.array([10.0, 50.0, 100.0, 500.0, 1000.0])
capped = soft_cap(logits)
mx.eval(capped)
print('Logit soft-capping (cap=50):')
for l, c in zip(logits.tolist(), capped.tolist()):
    print(f'  {l:>8.1f} → {c:>8.3f}')

## Comparison Table

| Feature | LLaMA 3 | Mistral | Gemma 2 |
|---------|---------|---------|--------|
| Params | 8B/70B | 7B | 2B/9B/27B |
| Context | 8K-128K | 32K | 8K |
| Attention | GQA | GQA + Sliding Window | GQA + Local/Global |
| Norm | RMSNorm | RMSNorm | RMSNorm |
| FFN | SwiGLU | SwiGLU | GeGLU |
| Position | RoPE | RoPE | RoPE |

**Next:** Notebook 10 — Metal Custom Kernels

---
## 🔬 Deep Dive: Gemma 4 Architecture (2025 SOTA)

Gemma 4 represents the current state-of-the-art in open model architecture. Let's implement its key innovations step by step.

### Gemma 4 Model Family

| Model | Params | Layers | d_model | Heads | KV Heads | Context | Type |
|-------|--------|--------|---------|-------|----------|---------|------|
| E2B | 2.3B (5.1B w/ emb) | 35 | 1536 | 2 | 1 | 128K | Dense + PLE |
| E4B | 4.5B (8B w/ emb) | 42 | 2560 | 2 | 1 | 128K | Dense + PLE |
| 31B | 30.7B | 60 | 2048 | 8 | 1 | 256K | Dense |
| 26B A4B | 25.2B | 30 | 1024 | 8 | 1 | 256K | MoE (128 experts, 8 active) |

### Key Innovations (vs Gemma 3)

1. **Per-Layer Embeddings (PLE)**: A second embedding table feeds residual signals into each layer
2. **K=V Sharing**: K and V are the same tensor in global attention layers
3. **p-RoPE**: Proportional RoPE (p=0.25) for global layers — better long-context
4. **Interleaved Local/Global**: Alternating sliding window and full attention
5. **Logit Soft-Capping**: `tanh(logits/cap) * cap` prevents attention score explosion
6. **MoE with Shared Expert**: 128 experts, 8 active, plus 1 always-on shared expert

In [ ]:
# Gemma 4 Innovation #1: Per-Layer Embeddings (PLE)
# 
# Standard transformers: one embedding table → same vector for each token across all layers
# PLE: TWO embedding tables → each layer gets a token-specific residual signal
#
# Why? Different layers need different information about each token.
# Early layers might need syntactic info, later layers need semantic info.

import mlx.core as mx
import mlx.nn as nn
import math

class PerLayerEmbedding(nn.Module):
    """Gemma 4's Per-Layer Embeddings (PLE).
    
    Adds a small, layer-specific residual to the hidden state at each layer.
    This lets each layer receive token-specific information without relying
    solely on the main embedding vector.
    """
    def __init__(self, vocab_size: int, d_model: int, n_layers: int):
        super().__init__()
        # Main embedding (standard)
        self.main_emb = nn.Embedding(vocab_size, d_model)
        # PLE: per-layer embedding table (smaller dimension, projected up)
        self.ple_emb = nn.Embedding(vocab_size, d_model)
        # Per-layer scaling factors (learned)
        self.layer_scales = [mx.zeros((1,)) for _ in range(n_layers)]
    
    def get_main_embedding(self, token_ids):
        """Get the main embedding (used once at input)."""
        return self.main_emb(token_ids)  # (batch, seq_len, d_model)
    
    def get_layer_residual(self, token_ids, layer_idx):
        """Get the per-layer residual signal."""
        ple = self.ple_emb(token_ids)  # (batch, seq_len, d_model)
        scale = mx.sigmoid(self.layer_scales[layer_idx])  # learned gate
        return ple * scale  # (batch, seq_len, d_model)

# Demo
vocab_size = 256
d_model = 64
n_layers = 4

ple = PerLayerEmbedding(vocab_size, d_model, n_layers)
token_ids = mx.array([[10, 20, 30, 40]])  # (1, 4)

main_emb = ple.get_main_embedding(token_ids)
print(f"Main embedding shape: {main_emb.shape}")  # (1, 4, 64)

for layer in range(n_layers):
    residual = ple.get_layer_residual(token_ids, layer)
    mx.eval(residual)
    print(f"Layer {layer} PLE residual norm: {mx.sqrt(mx.sum(residual * residual)).item():.4f}")

print("\n💡 PLE lets each layer 'ask' for different token information.")
print("   Early layers might use PLE for syntax, later layers for semantics.")
print("   The learned scale gates control how much PLE signal each layer receives.")

The next cell continues the implementation:

**Gemma 4 Innovation #2: p-RoPE (Proportional RoPE)**

In [ ]:
# Gemma 4 Innovation #2: p-RoPE (Proportional RoPE)
#
# Standard RoPE: theta_base = 10000 for ALL layers
# p-RoPE: theta_base = 10000 for local layers, but scaled by factor p for global layers
#
# Why? Global attention needs to "see" further. Lower frequencies = longer range.
# p=0.25 means global layers use 4x lower frequencies → 4x longer effective range.

def precompute_rope_freqs(d_head: int, max_seq: int, theta: float = 10000.0, p: float = 1.0):
    """Precompute RoPE frequencies with optional proportional scaling (p-RoPE).
    
    Args:
        d_head: head dimension
        max_seq: max sequence length
        theta: base frequency
        p: proportional scaling factor (1.0 = standard RoPE, 0.25 = Gemma 4 global)
    """
    # Scale the base frequency by p
    effective_theta = theta / p  # Lower p → higher theta → lower frequencies → longer range
    
    i = mx.arange(0, d_head, 2)
    freqs = 1.0 / (effective_theta ** (i / d_head))
    pos = mx.arange(max_seq)
    angles = pos.reshape(-1, 1) * freqs.reshape(1, -1)
    
    cos_f = mx.cos(angles)
    sin_f = mx.sin(angles)
    mx.eval(cos_f, sin_f)
    return cos_f, sin_f

# Compare standard RoPE vs p-RoPE frequencies
import numpy as np
import matplotlib.pyplot as plt

d_head = 16
max_seq = 256

cos_standard, _ = precompute_rope_freqs(d_head, max_seq, p=1.0)
cos_prope, _ = precompute_rope_freqs(d_head, max_seq, p=0.25)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.imshow(np.array(cos_standard), aspect='auto', cmap='RdBu_r')
ax1.set_title('Standard RoPE (p=1.0)\nLocal Attention Layers')
ax1.set_xlabel('Frequency index')
ax1.set_ylabel('Position')

ax2.imshow(np.array(cos_prope), aspect='auto', cmap='RdBu_r')
ax2.set_title('p-RoPE (p=0.25)\nGlobal Attention Layers')
ax2.set_xlabel('Frequency index')
ax2.set_ylabel('Position')

plt.suptitle('Gemma 4: Standard RoPE vs p-RoPE', fontsize=12)
plt.tight_layout()
plt.show()

print("💡 p-RoPE (right) has SLOWER oscillations → captures LONGER-range positions")
print("   This is why Gemma 4 can handle 256K context: global layers 'see' further")
print("   Local layers use standard RoPE (fast oscillations for nearby tokens)")

The next cell continues the implementation:

**Gemma 4 Innovation #3: Mixture of Experts (MoE)**

In [ ]:
# Gemma 4 Innovation #3: Mixture of Experts (MoE)
#
# The 26B A4B model uses MoE: 128 total experts, 8 active per token, plus 1 shared expert
# Only ~4B parameters are active per token → efficiency of a 4B model, quality of a 26B model
#
# This is the SAME principle used by Mixtral, DeepSeek-V3, and other frontier models

class MoERouter(nn.Module):
    """Top-K expert router for Mixture of Experts."""
    def __init__(self, d_model: int, n_experts: int, top_k: int = 2):
        super().__init__()
        self.gate = nn.Linear(d_model, n_experts, bias=False)
        self.top_k = top_k
        self.n_experts = n_experts
    
    def __call__(self, x: mx.array):
        """Route each token to top-k experts.
        
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            expert_weights: (batch, seq_len, top_k) — normalized weights
            expert_indices: (batch, seq_len, top_k) — which experts to use
        """
        logits = self.gate(x)  # (batch, seq_len, n_experts)
        # Get top-k indices via argsort (descending), take first k
        sorted_indices = mx.argsort(-logits, axis=-1)  # descending
        expert_indices = sorted_indices[..., :self.top_k]  # (B, T, top_k)
        
        # Gather the logits for the selected experts
        # Use advanced indexing to pick the top-k logit values
        B, T, _ = logits.shape
        b_idx = mx.arange(B).reshape(B, 1, 1)
        t_idx = mx.arange(T).reshape(1, T, 1)
        top_k_logits = logits[b_idx, t_idx, expert_indices]  # (B, T, top_k)
        
        expert_weights = mx.softmax(top_k_logits, axis=-1)
        return expert_weights, expert_indices


class MoEFFN(nn.Module):
    """Mixture of Experts Feed-Forward Network (Gemma 4 style).
    
    - n_experts total experts (e.g., 128)
    - top_k active per token (e.g., 8)
    - 1 shared expert (always active)
    """
    def __init__(self, d_model: int, d_ff: int, n_experts: int = 8, top_k: int = 2):
        super().__init__()
        self.router = MoERouter(d_model, n_experts, top_k)
        self.experts_up = [nn.Linear(d_model, d_ff, bias=False) for _ in range(n_experts)]
        self.experts_down = [nn.Linear(d_ff, d_model, bias=False) for _ in range(n_experts)]
        # Shared expert (always active, Gemma 4 innovation)
        self.shared_up = nn.Linear(d_model, d_ff, bias=False)
        self.shared_down = nn.Linear(d_ff, d_model, bias=False)
        self.top_k = top_k
        self.n_experts = n_experts
    
    def __call__(self, x: mx.array) -> mx.array:
        B, T, D = x.shape
        
        # Route tokens to experts
        expert_weights, expert_indices = self.router(x)  # (B,T,k), (B,T,k)
        
        # Shared expert output (always computed)
        shared_out = self.shared_down(nn.silu(self.shared_up(x)))  # (B, T, D)
        
        # Compute selected expert outputs and combine with routing weights
        # (Simplified loop over top-k — production code uses sparse/batched ops)
        routed_out = mx.zeros_like(x)  # (B, T, D)
        for k_idx in range(self.top_k):
            w = expert_weights[..., k_idx:k_idx+1]  # (B, T, 1)
            e_idx = expert_indices[..., k_idx]        # (B, T) — which expert
            
            # For each expert, compute output for tokens routed to it
            for e in range(self.n_experts):
                mask = (e_idx == e).astype(x.dtype).reshape(B, T, 1)  # (B, T, 1)
                if mx.sum(mask).item() > 0:
                    expert_out = self.experts_down[e](nn.silu(self.experts_up[e](x)))
                    routed_out = routed_out + w * mask * expert_out
        
        output = shared_out + routed_out  # Combine shared + routed
        
        print(f"  Router selected top-{self.top_k} of {self.n_experts} experts per token")
        print(f"  + 1 shared expert (always active)")
        print(f"  Active params per token: ~{(self.top_k + 1)}/{self.n_experts + 1} of total")
        
        return output  # (B, T, D)


# Demo
d_model = 64
d_ff = 256
n_experts = 8  # Gemma 4 uses 128, we use 8 for demo
top_k = 2      # Gemma 4 uses 8

moe = MoEFFN(d_model, d_ff, n_experts=n_experts, top_k=top_k)
x = mx.random.normal((1, 4, d_model))
out = moe(x)
mx.eval(out)

print(f"\nInput:  {x.shape}")
print(f"Output: {out.shape}")

# Parameter comparison: Dense vs MoE
dense_params = d_model * d_ff * 2  # up + down
moe_total_params = n_experts * d_model * d_ff * 2 + d_model * d_ff * 2  # experts + shared
moe_active_params = (top_k + 1) * d_model * d_ff * 2

print(f"\n--- Parameter Comparison ---")
print(f"Dense FFN:        {dense_params:>10,} params (all active)")
print(f"MoE total:        {moe_total_params:>10,} params")
print(f"MoE active/token: {moe_active_params:>10,} params ({moe_active_params/moe_total_params*100:.0f}% of total)")
print(f"\n🎯 MoE gives {moe_total_params/moe_active_params:.1f}x more total knowledge")
print(f"   while only using {moe_active_params/dense_params:.1f}x the compute of a dense model")

## 🎯 Key Takeaways

1. **The Modern LLM Recipe is settled.** Nearly every competitive open model (LLaMA, Mistral, Gemma, Qwen, DeepSeek) uses the same core stack: **RMSNorm + SwiGLU + RoPE + GQA**. If you understand these four components, you understand the backbone of every frontier LLM.

2. **Sliding window attention** (Mistral) trades global context for O(n×W) efficiency. Interleaving local and global layers (Gemma) gives you the best of both worlds.

3. **Logit soft-capping** (`tanh(x/cap) * cap`) is a simple but effective trick from Gemma 2 that prevents attention scores from exploding — no extra parameters needed.

4. **The frontier is MoE for efficiency.** Mixture of Experts lets models have many more total parameters while keeping per-token compute fixed. DeepSeek-V3 (671B total, 37B active) trained a GPT-4-class model for ~$5.5M. Gemma 4's MoE variant (26B total, ~4B active) brings this to smaller scales.

5. **Architecture innovations compound.** Gemma 4 combines PLE (per-layer embeddings), p-RoPE (proportional frequencies for long context), K=V sharing, and MoE with a shared expert — each small idea stacks into a meaningfully better model.

## 🧪 Try It Yourself

Test your understanding of modern architectures:

1. **Sliding window**: Change the window size from 4 to 8 to 16. How does the attention pattern change? At what window size does it look like full attention?

2. **Soft-capping**: Try different cap values (10, 50, 100, 1000). What happens when the cap is very small? Very large?

3. **Architecture quiz**: Without looking, list the 5 key differences between a vanilla transformer and LLaMA. Check your answer against the comparison table.

---
## ➡️ What's Next?

You've completed this notebook! In **Notebook 10: Metal Custom Kernels**, we'll explore writing GPU kernels for maximum performance.

💡 **Before moving on**, make sure you can answer these questions:
- What was the main concept in this notebook?
- Why does it matter for building LLMs?
- Could you explain it to a friend in one sentence?

## 📜 History & Alternatives

### The Evolution of Language Model Architectures

The architecture landscape has evolved from simple recurrence to the transformer paradigm, with each generation solving a key limitation of its predecessor.

| Year | Architecture | Team | Key Contribution |
|------|-------------|------|-----------------|
| 2017 | **Transformer** | Vaswani et al. (Google) | Self-attention replaces recurrence — parallel training, "Attention Is All You Need" |
| 2018 | **GPT-1** | Radford et al. (OpenAI) | Decoder-only transformer + unsupervised pretraining → fine-tuning paradigm |
| 2018 | **BERT** | Devlin et al. (Google) | Encoder-only, bidirectional — masked language modeling, dominated NLU benchmarks |
| 2019 | **GPT-2** | Radford et al. (OpenAI) | Scaled up GPT-1 (1.5B params) — showed emergent zero-shot abilities |
| 2019 | **T5** | Raffel et al. (Google) | Encoder-decoder, text-to-text framework — unified all NLP tasks |
| 2020 | **GPT-3** | Brown et al. (OpenAI) | 175B params — in-context learning, few-shot prompting without fine-tuning |
| 2022 | **Chinchilla** | Hoffmann et al. (DeepMind) | Proved smaller models + more data beats larger models + less data |
| 2023 | **LLaMA** | Touvron et al. (Meta) | Open-weight, efficient (RMSNorm, SwiGLU, RoPE) — launched open-source LLM era |
| 2023 | **Mistral 7B** | Mistral AI | Sliding window attention + GQA — outperformed LLaMA 2 13B at half the size |
| 2023 | **Mixtral 8x7B** | Mistral AI | Sparse MoE — 8 experts, 2 active, matched GPT-3.5 quality |
| 2024 | **LLaMA 3** | Meta AI | 8B/70B/405B, 128K context, GQA — new open-source SOTA |
| 2024 | **Gemma 2** | Google DeepMind | Knowledge distillation from Gemini, sliding window + global attention |
| 2024 | **Qwen 2.5** | Alibaba | Strong multilingual, 0.5B to 72B, competitive with LLaMA 3 |
| 2025 | **DeepSeek-V3** | DeepSeek | MoE (256 experts), multi-token prediction, trained for $5.5M |
| 2025 | **DeepSeek-R1** | DeepSeek | Reasoning model via RL — open-weight alternative to o1 |
| 2025 | **LLaMA 4** | Meta AI | MoE architecture (Scout/Maverick), 10M token context |
| 2025 | **Gemma 3** | Google DeepMind | Multimodal, 128K context, ShieldGemma safety |
| 2025 | **Gemma 4** | Google DeepMind | Diffusion-based, MoE with shared experts, advanced reasoning |

### 💡 The Three Architecture Paradigms

| Paradigm | Architecture | Training Objective | Best For |
|----------|-------------|-------------------|----------|
| **Encoder-only** | BERT, RoBERTa | Masked LM (bidirectional) | Classification, NER, retrieval |
| **Decoder-only** | GPT, LLaMA, Mistral | Causal LM (left-to-right) | Text generation, chat, reasoning |
| **Encoder-Decoder** | T5, BART | Seq2seq (span corruption) | Translation, summarization |

The field has converged on **decoder-only** for general-purpose LLMs because:
1. Simpler architecture (one stack, not two)
2. Naturally supports generation
3. In-context learning emerges at scale
4. Easier to scale training

### The Modern LLM Recipe (2024-2025)

Most competitive open LLMs share these design choices:
- **Architecture**: Decoder-only transformer with GQA
- **Normalization**: RMSNorm (pre-norm)
- **Activation**: SwiGLU in FFN
- **Positional encoding**: RoPE (often with extensions for long context)
- **Vocabulary**: 32K-128K BPE tokens
- **Training**: Multi-stage (pretraining → SFT → RLHF/DPO)

### ⚡ The Efficiency Revolution

DeepSeek-V3 trained a GPT-4-class model for ~$5.5M (vs estimated $100M+ for GPT-4). Key efficiency innovations:
- MoE: 671B total params but only 37B active per token
- FP8 mixed precision training
- Multi-token prediction auxiliary objective
- Efficient data pipeline and curriculum

### ⚠️ Common Pitfalls

1. **Assuming bigger = better**: Chinchilla (2022) proved that a 70B model trained on 1.4T tokens beats a 280B model trained on 300B tokens. Training data quantity matters as much as parameter count.
2. **Ignoring the encoder-decoder option**: Decoder-only dominates for chat/generation, but encoder-decoder (T5-style) can still be superior for structured tasks like translation and summarization. Don't default to decoder-only without considering the task.
3. **Confusing total params with active params in MoE**: Mixtral 8x7B has ~47B total parameters but only ~13B active per token. Comparing it to a dense 47B model on inference cost is misleading — it runs closer to a 13B model in speed and memory.
4. **Overlooking sliding window attention limits**: Mistral's sliding window is efficient but means tokens beyond the window can only attend to each other indirectly through intermediate layers. For tasks requiring precise long-range retrieval, global attention layers (as in Gemma 2/3) are needed.
5. **Treating architecture choices as independent**: RMSNorm, SwiGLU, RoPE, and GQA interact — e.g., RoPE's rotation frequencies must be tuned differently with GQA vs MHA. Swapping one component without re-tuning others can degrade performance.

### 🎯 Interview Tip

> *"The decoder-only architecture won for LLMs because of a surprising property: causal language modeling (predicting the next token) is a universal task that implicitly learns to solve classification, translation, reasoning, and more — all through in-context learning. BERT-style bidirectional models are better at understanding but can't generate. The modern recipe (RMSNorm + SwiGLU + RoPE + GQA) was largely established by LLaMA in 2023 and has become the de facto standard. The frontier is now MoE (DeepSeek-V3, Gemma 4) for better compute efficiency."*